# 🧠 Detección temprana de rasgos del espectro autista en niños pequeños con Machine Learning

**Proyecto final – Machine Learning** · Maestría en Inteligencia Artificial, Educación e Investigación
**Autora:** Francis Melissa Paredes Hernández
**Tipo de problema:** Clasificación binaria supervisada
**Herramientas:** Python, pandas, scikit-learn, seaborn, ydata-profiling, SQLite y un simulador en HTML

---
<a id="toc"></a>
## 📑 Tabla de contenido

1. [Sumario del proyecto](#s1)
2. [Descripción detallada del proyecto](#s2)
3. [Preparación del entorno](#s3)
4. [Ingesta de datos y ELT](#s4)
5. [Reporte de Data Profiling](#s5)
6. [Análisis exploratorio de datos (EDA)](#s6)
7. [Preparación para el modelado](#s7)
8. [Entrenamiento y comparación de modelos](#s8)
9. [Evaluación del mejor modelo](#s9)
10. [Importancia de las variables](#s10)
11. [Predicciones](#s11)
12. [Explicación de las variables (tabla)](#s12)
13. [Simulador interactivo en HTML](#s13)
14. [Conclusiones, limitaciones y próximos pasos](#s14)
15. [Referencias](#s15)

<a id="s1"></a>
## 1. Sumario del proyecto

**Problemática.** Los rasgos del espectro autista (TEA) pueden identificarse de forma temprana mediante cuestionarios breves de tamizaje, pero su revisión depende de especialistas que no siempre están disponibles. Una herramienta de apoyo que priorice casos para evaluación profesional puede ayudar a familias y a docentes.

**Solución propuesta.** Un modelo de clasificación que, a partir de las respuestas a un cuestionario de 10 preguntas (Q-CHAT-10) y algunos datos básicos del niño, estima la probabilidad de que presente rasgos del espectro autista.

**Qué hace este cuaderno, en resumen:**

| Etapa | Resultado |
|---|---|
| Ingesta y ELT | Datos cargados en una base SQLite y transformados a un formato limpio |
| Data Profiling | Reporte HTML automático con `ydata-profiling` |
| EDA | Gráficos y hallazgos sobre balance de clases, edad, ítems y correlaciones |
| Modelado | Comparación de 3 algoritmos con validación cruzada |
| Evaluación | Métricas, matriz de confusión y curva ROC |
| Predicciones | Tabla de casos de prueba y predicción de un caso nuevo |
| Simulador | Página HTML interactiva que calcula la probabilidad en el navegador |

> ⚠️ **Aviso importante:** este proyecto es un ejercicio académico y educativo. Un modelo de tamizaje **no es un diagnóstico**; cualquier resultado debe ser valorado por profesionales de la salud.

<a id="s2"></a>
## 2. Descripción detallada del proyecto

### 2.1 Planteamiento del problema
La detección temprana de condiciones del neurodesarrollo mejora las oportunidades de intervención y apoyo educativo. Sin embargo, el acceso a evaluaciones especializadas es limitado. ¿Es posible construir un modelo sencillo que, con las respuestas de un cuestionario de tamizaje, identifique qué casos merecen una evaluación prioritaria?

### 2.2 Objetivo general
Desarrollar un prototipo de Machine Learning que clasifique, a partir de datos de tamizaje, si un niño pequeño presenta rasgos del espectro autista, como apoyo a la orientación temprana.

### 2.3 Objetivos específicos
1. Ingerir los datos y aplicar un proceso **ELT** (extraer, cargar y transformar) reproducible.
2. Generar un reporte de **Data Profiling** para conocer la calidad de los datos.
3. Realizar un **análisis exploratorio** que explique el comportamiento de las variables.
4. Entrenar y comparar tres modelos de clasificación y seleccionar el mejor con criterios justificados.
5. Evaluar el modelo con métricas apropiadas para un contexto de salud (en particular el *recall*).
6. Construir un **simulador en HTML** que permita interactuar con el modelo.

### 2.4 Conjunto de datos
- **Nombre:** *Autism Screening Data for Toddlers* (tamizaje de autismo en niños pequeños), publicado por Fadi Thabtah y disponible en Kaggle.
- **Contenido:** una fila por niño, con 10 ítems de tamizaje (`A1` a `A10`, codificados 0 o 1), edad en meses, sexo, antecedentes (ictericia, familiar con TEA), quién completó el cuestionario y la clase final (rasgos TEA: Yes/No).
- **Cómo usarlo:** descarga el CSV desde Kaggle y colócalo junto a este cuaderno con el nombre `Toddler Autism dataset July 2018.csv` (o sube el archivo en Colab).
- **Si el archivo no está disponible**, el cuaderno genera automáticamente un conjunto **sintético** con la misma estructura para que puedas ejecutar todo el flujo. Los resultados con datos sintéticos son solo demostrativos y **no deben presentarse como resultados reales**.

### 2.5 Decisiones éticas de diseño
- Se **excluye la etnicidad** del modelo para evitar sesgos innecesarios.
- Se **excluye la puntuación total** (`qchat_10_score`) porque es la suma de los ítems y la clase se deriva de ella (fuga de información). Más adelante se explica con un experimento.
- El resultado se presenta como **probabilidad de apoyo a la decisión**, nunca como diagnóstico.

### 2.6 Metodología (CRISP-DM simplificada)
Comprensión del problema → ingesta y ELT → profiling y EDA → preparación → modelado → evaluación → despliegue (simulador).

<a id="s3"></a>
## 3. Preparación del entorno

Instalamos la librería de *profiling* (solo la primera vez) e importamos las herramientas que usaremos. En Google Colab, el resto de librerías ya viene instalado.

In [ ]:
# Instalación de ydata-profiling (herramienta que genera el reporte de Data Profiling).
# Si ya la tienes instalada, puedes omitir esta celda.
%pip install -q ydata-profiling

In [ ]:
# ===== Importación de librerías =====
import os, json, sqlite3, html, warnings

import numpy as np                      # cálculo numérico
import pandas as pd                     # manejo de tablas de datos
import matplotlib.pyplot as plt         # gráficos básicos
import seaborn as sns                   # gráficos estadísticos

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, classification_report,
                             RocCurveDisplay)
from sklearn.inspection import permutation_importance
import joblib                           # para guardar el modelo entrenado

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

SEMILLA = 42                            # semilla para que los resultados sean reproducibles
np.random.seed(SEMILLA)
print("Entorno listo ✅")

<a id="s4"></a>
## 4. Ingesta de datos y ELT

**ELT** significa *Extract, Load, Transform*: primero **extraemos** los datos crudos, luego los **cargamos** tal cual en un almacén (aquí, una base SQLite) y finalmente los **transformamos**. Así conservamos siempre una copia intacta del dato original.

### 4.1 Extract: lectura de los datos

In [ ]:
# Nombre del archivo descargado de Kaggle
RUTA_CSV = "Toddler Autism dataset July 2018.csv"

# En Google Colab puedes subir el archivo descomentando estas dos líneas:
# from google.colab import files
# files.upload()

def generar_datos_sinteticos(n=1054, semilla=SEMILLA):
    """Crea un conjunto SINTÉTICO con la misma estructura del dataset real.
    Solo sirve para probar el flujo cuando no se tiene el archivo original."""
    rng = np.random.default_rng(semilla)
    con_rasgos = rng.random(n) < 0.65                           # clase latente
    prob_base = np.where(con_rasgos, 0.70, 0.25)                # prob. de señal de alerta
    ajuste_item = rng.uniform(-0.10, 0.10, 10)                  # cada ítem es un poco distinto
    prob = np.clip(prob_base[:, None] + ajuste_item, 0.02, 0.98)
    A = (rng.random((n, 10)) < prob).astype(int)
    puntaje = A.sum(axis=1)
    df = pd.DataFrame(A, columns=[f"A{i}" for i in range(1, 11)])
    df.insert(0, "Case_No", np.arange(1, n + 1))
    df["Age_Mons"] = rng.integers(12, 37, n)
    df["Qchat-10-Score"] = puntaje
    df["Sex"] = rng.choice(["m", "f"], n, p=[0.7, 0.3])
    df["Ethnicity"] = rng.choice(["asian", "black", "middle eastern", "white european", "latino", "others"], n)
    df["Jaundice"] = rng.choice(["yes", "no"], n, p=[0.28, 0.72])
    df["Family_mem_with_ASD"] = rng.choice(["yes", "no"], n, p=[0.17, 0.83])
    df["Who completed the test"] = rng.choice(["family member", "health care professional", "self"], n, p=[0.9, 0.08, 0.02])
    df["Class/ASD Traits "] = np.where(puntaje > 3, "Yes", "No")  # regla del dataset original
    return df

if os.path.exists(RUTA_CSV):
    df_raw = pd.read_csv(RUTA_CSV)
    FUENTE = "REAL (archivo de Kaggle)"
else:
    df_raw = generar_datos_sinteticos()
    FUENTE = "SINTÉTICA (solo demostrativa)"

print(f"Fuente de datos: {FUENTE}")
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.head()

### 4.2 Load: carga de los datos crudos en SQLite
Guardamos la tabla **sin modificar** en una base de datos. SQLite viene incluida en Python y no requiere instalación.

In [ ]:
# Abrimos (o creamos) la base de datos local
conexion = sqlite3.connect("proyecto_ml.db")

# Cargamos la tabla cruda tal cual llegó
df_raw.to_sql("tamizaje_raw", conexion, if_exists="replace", index=False)

# Verificamos con una consulta SQL
pd.read_sql("SELECT COUNT(*) AS total_filas FROM tamizaje_raw", conexion)

### 4.3 Transform: limpieza y transformación
Pasos que aplicaremos:
1. Leer la tabla cruda desde la base con SQL.
2. Estandarizar los nombres de las columnas (minúsculas, sin espacios).
3. Convertir los textos en valores numéricos (0/1) para que el modelo pueda usarlos.
4. Revisar valores nulos y duplicados.

In [ ]:
# 1) Leemos desde la base de datos con SQL
df = pd.read_sql("SELECT * FROM tamizaje_raw", conexion)

# 2) Estandarizamos los nombres de columnas
def limpiar_nombre(col):
    col = col.strip().lower()
    for viejo, nuevo in [(" ", "_"), ("/", "_"), ("-", "_")]:
        col = col.replace(viejo, nuevo)
    return col

df.columns = [limpiar_nombre(c) for c in df.columns]
print("Columnas:", list(df.columns))

# 3) Convertimos textos a números
col_clase = [c for c in df.columns if c.startswith("class")][0]   # columna objetivo original
df["asd"] = (df[col_clase].str.strip().str.lower() == "yes").astype(int)   # 1 = presenta rasgos
df["sex_m"] = (df["sex"].str.strip().str.lower() == "m").astype(int)       # 1 = niño, 0 = niña
df["jaundice"] = (df["jaundice"].str.strip().str.lower() == "yes").astype(int)
df["family_mem_with_asd"] = (df["family_mem_with_asd"].str.strip().str.lower() == "yes").astype(int)

# 4) Calidad de los datos
print("\nValores nulos por columna:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() > 0 else "No hay valores nulos ✅")
print("\nFilas duplicadas:", df.duplicated().sum())

# Guardamos la tabla limpia también en la base de datos (parte "T" del ELT)
df.to_sql("tamizaje_limpia", conexion, if_exists="replace", index=False)
df.head()

<a id="s5"></a>
## 5. Reporte de Data Profiling

El **Data Profiling** es un diagnóstico automático de la calidad de los datos: tipos de variables, valores faltantes, distribuciones, correlaciones y alertas. Usaremos **ydata-profiling**, que genera un reporte HTML completo con una sola instrucción. El archivo se guarda como `reporte_data_profiling.html` y puedes incluirlo en tu repositorio.

In [ ]:
try:
    from ydata_profiling import ProfileReport

    columnas_perfil = ["a1","a2","a3","a4","a5","a6","a7","a8","a9","a10",
                       "age_mons","qchat_10_score","sex_m","jaundice",
                       "family_mem_with_asd","ethnicity","asd"]
    columnas_perfil = [c for c in columnas_perfil if c in df.columns]

    perfil = ProfileReport(df[columnas_perfil],
                           title="Data Profiling – Tamizaje de rasgos TEA en niños pequeños",
                           minimal=False)
    perfil.to_file("reporte_data_profiling.html")          # guarda el reporte
    print("Reporte guardado en: reporte_data_profiling.html ✅")
    perfil.to_notebook_iframe()                            # lo muestra dentro del cuaderno
except ImportError:
    print("Instala la librería con: %pip install ydata-profiling  y vuelve a ejecutar esta celda.")

<a id="s6"></a>
## 6. Análisis exploratorio de datos (EDA)

El EDA nos ayuda a **entender los datos antes de modelar**. Responderemos estas preguntas:
- ¿Cuántos niños hay en cada clase? (balance)
- ¿Cómo se distribuyen la edad y la puntuación del cuestionario?
- ¿Qué ítems se asocian más con la presencia de rasgos?
- ¿Influyen el sexo, la ictericia o los antecedentes familiares?

### 6.1 Resumen estadístico

In [ ]:
# Información general de la tabla
df.info()

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df[["age_mons", "qchat_10_score"]].describe().round(2)

### 6.2 Balance de las clases
Si una clase fuera muy escasa, el modelo podría ignorarla. Revisamos cuántos casos hay de cada una.

In [ ]:
conteo = df["asd"].value_counts().sort_index()
porcentaje = (conteo / len(df) * 100).round(1)
print(pd.DataFrame({"casos": conteo, "porcentaje_%": porcentaje}).rename(index={0: "Sin rasgos", 1: "Con rasgos"}))

plt.figure(figsize=(5, 3.5))
ax = sns.countplot(x="asd", data=df, palette=["#4C9BE8", "#F28C6B"])
ax.set_xticklabels(["Sin rasgos (0)", "Con rasgos (1)"])
ax.set_title("Distribución de la clase objetivo")
ax.set_xlabel(""); ax.set_ylabel("Número de niños")
plt.show()

### 6.3 Edad y puntuación del cuestionario

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="age_mons", hue="asd", bins=12, multiple="stack",
             palette=["#4C9BE8", "#F28C6B"], ax=ejes[0])
ejes[0].set_title("Edad (en meses) según clase")

sns.countplot(data=df, x="qchat_10_score", hue="asd", palette=["#4C9BE8", "#F28C6B"], ax=ejes[1])
ejes[1].set_title("Puntuación Q-CHAT-10 según clase")

plt.tight_layout(); plt.show()

### 6.4 ¿Qué ítems se asocian más con la clase?
Calculamos, para cada ítem, el **porcentaje de niños con señal de alerta (valor 1)** dentro de cada clase. Mientras mayor sea la diferencia, más informativo es el ítem.

In [ ]:
items = [f"a{i}" for i in range(1, 11)]
tasa_items = df.groupby("asd")[items].mean().T * 100
tasa_items.columns = ["Sin rasgos", "Con rasgos"]
tasa_items["diferencia"] = (tasa_items["Con rasgos"] - tasa_items["Sin rasgos"]).round(1)
display(tasa_items.round(1))

tasa_items[["Sin rasgos", "Con rasgos"]].plot(kind="bar", figsize=(10, 4), color=["#4C9BE8", "#F28C6B"])
plt.title("% de señales de alerta por ítem y clase")
plt.ylabel("% con valor 1"); plt.xticks(rotation=0)
plt.show()

### 6.5 Variables demográficas y antecedentes

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.8))
for eje, (col, titulo) in zip(ejes, [("sex_m", "Sexo (1 = niño)"),
                                    ("jaundice", "Ictericia al nacer"),
                                    ("family_mem_with_asd", "Familiar con TEA")]):
    tasa = df.groupby(col)["asd"].mean() * 100
    sns.barplot(x=tasa.index, y=tasa.values, color="#7FB77E", ax=eje)
    eje.set_title(titulo); eje.set_ylabel("% con rasgos"); eje.set_xlabel("0 = No / 1 = Sí")
plt.tight_layout(); plt.show()

### 6.6 Correlaciones
Un mapa de calor muestra qué variables se mueven juntas. Fíjate en que `qchat_10_score` correlaciona muy fuerte con la clase: es una **pista de fuga de información** que trataremos en la sección 7.

In [ ]:
columnas_corr = items + ["age_mons", "sex_m", "jaundice", "family_mem_with_asd", "qchat_10_score", "asd"]
plt.figure(figsize=(11, 8))
sns.heatmap(df[columnas_corr].corr(), cmap="coolwarm", center=0, annot=False, linewidths=0.3)
plt.title("Matriz de correlación")
plt.show()

print("Correlación de cada variable con la clase (ordenada):")
df[columnas_corr].corr()["asd"].drop("asd").sort_values(ascending=False).round(2)

**Cómo interpretar el EDA (completa con tus propios hallazgos al ejecutar con los datos reales):**
- Revisa el porcentaje de cada clase y si el conjunto está desbalanceado.
- Identifica los ítems con mayor diferencia entre clases.
- Comenta si la edad, el sexo, la ictericia o los antecedentes familiares muestran diferencias claras.

<a id="s7"></a>
## 7. Preparación para el modelado

### 7.1 Selección de variables
- **Variables predictoras (X):** los 10 ítems `a1`–`a10`, `age_mons`, `sex_m`, `jaundice` y `family_mem_with_asd`.
- **Variable objetivo (y):** `asd` (1 = presenta rasgos, 0 = no).
- **Excluidas:**
  - `qchat_10_score`: es la suma de los ítems y la clase se define a partir de ella (**fuga de información**).
  - `ethnicity` y `who_completed_the_test`: no aportan de forma ética ni metodológicamente defendible.

### 7.2 División entrenamiento / prueba
Separamos 80 % para entrenar y 20 % para probar. Usamos `stratify=y` para mantener la proporción de clases en ambos grupos.

In [ ]:
VARIABLES = items + ["age_mons", "sex_m", "jaundice", "family_mem_with_asd"]

X = df[VARIABLES]
y = df["asd"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Proporción de la clase 1 en entrenamiento:", round(y_train.mean(), 3))
print("Proporción de la clase 1 en prueba:       ", round(y_test.mean(), 3))

### 7.3 Preprocesamiento
Las variables binarias (0/1) no necesitan cambios. La edad en meses tiene otra escala, así que la **estandarizamos** (media 0 y desviación 1). Esto es necesario para la regresión logística y no perjudica a los modelos de árboles. Lo hacemos dentro de un `Pipeline` para evitar fugas entre entrenamiento y prueba.

In [ ]:
# Estandariza solo la edad; el resto de columnas pasa sin cambios
preprocesador = ColumnTransformer(
    transformers=[("num", StandardScaler(), ["age_mons"])],
    remainder="passthrough")

<a id="s8"></a>
## 8. Entrenamiento y comparación de modelos

Compararemos tres algoritmos de clasificación con distinto nivel de complejidad:

| Modelo | Idea básica | Ventaja |
|---|---|---|
| **Regresión logística** | Combina las variables con pesos y devuelve una probabilidad | Simple e interpretable |
| **Random Forest** | Muchos árboles de decisión que votan | Robusto, captura relaciones no lineales |
| **Gradient Boosting** | Árboles que corrigen los errores de los anteriores | Suele ser muy preciso |

Usaremos **validación cruzada estratificada de 5 pliegues**: el entrenamiento se divide en 5 partes, y cada parte se usa una vez para validar. Así evaluamos el modelo de forma más confiable que con una sola división.

In [ ]:
# Definimos los modelos dentro de un Pipeline (preprocesamiento + clasificador)
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000, random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEMILLA),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEMILLA),
}
pipelines = {nombre: Pipeline([("prep", preprocesador), ("clf", clf)])
             for nombre, clf in modelos.items()}

# Validación cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
metricas_cv = {"f1": "f1", "recall": "recall", "precision": "precision", "roc_auc": "roc_auc"}

filas = []
for nombre, pipe in pipelines.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=metricas_cv)
    filas.append({"Modelo": nombre,
                  "F1 (CV)": res["test_f1"].mean(),
                  "Recall (CV)": res["test_recall"].mean(),
                  "Precisión (CV)": res["test_precision"].mean(),
                  "ROC-AUC (CV)": res["test_roc_auc"].mean()})

tabla_cv = pd.DataFrame(filas).set_index("Modelo").round(3)
tabla_cv.sort_values("F1 (CV)", ascending=False)

### 8.1 Entrenamiento final y evaluación en el conjunto de prueba
Entrenamos cada modelo con **todo** el conjunto de entrenamiento y lo medimos sobre datos que nunca vio (el 20 % de prueba).

**Métricas:**
- **Accuracy:** porcentaje total de aciertos.
- **Precisión:** de los que el modelo marcó como "con rasgos", cuántos realmente lo eran.
- **Recall (sensibilidad):** de los niños que sí tenían rasgos, cuántos detectó el modelo. En salud es la más importante, porque un falso negativo es costoso.
- **F1:** equilibrio entre precisión y recall.
- **ROC-AUC:** capacidad del modelo para separar las dos clases (1.0 es perfecto, 0.5 es azar).

In [ ]:
resultados = []
for nombre, pipe in pipelines.items():
    pipe.fit(X_train, y_train)                         # entrenamiento
    pred = pipe.predict(X_test)                        # clase predicha
    prob = pipe.predict_proba(X_test)[:, 1]            # probabilidad de la clase 1
    resultados.append({"Modelo": nombre,
                       "Accuracy": accuracy_score(y_test, pred),
                       "Precisión": precision_score(y_test, pred),
                       "Recall": recall_score(y_test, pred),
                       "F1": f1_score(y_test, pred),
                       "ROC-AUC": roc_auc_score(y_test, prob)})

tabla_test = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabla_test.sort_values("F1", ascending=False)

In [ ]:
# Elegimos el mejor modelo según el F1 en el conjunto de prueba
nombre_mejor = tabla_test["F1"].idxmax()
mejor_modelo = pipelines[nombre_mejor]
print(f"🏆 Mejor modelo: {nombre_mejor}")
print(tabla_test.loc[nombre_mejor].to_dict())

<a id="s9"></a>
## 9. Evaluación del mejor modelo

### 9.1 Matriz de confusión y curva ROC
- La **matriz de confusión** muestra los aciertos y los tipos de error: falsos positivos (se marcó un niño sin rasgos) y falsos negativos (se dejó pasar un niño con rasgos).
- La **curva ROC** resume el desempeño para todos los umbrales posibles.

In [ ]:
pred_mejor = mejor_modelo.predict(X_test)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_mejor, display_labels=["Sin rasgos", "Con rasgos"],
    cmap="Blues", ax=ejes[0])
ejes[0].set_title(f"Matriz de confusión – {nombre_mejor}")

RocCurveDisplay.from_estimator(mejor_modelo, X_test, y_test, ax=ejes[1])
ejes[1].plot([0, 1], [0, 1], "k--", label="Azar")
ejes[1].set_title("Curva ROC"); ejes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
print(classification_report(y_test, pred_mejor, target_names=["Sin rasgos", "Con rasgos"]))

### 9.2 Experimento: ¿qué pasa si no usamos los ítems del cuestionario?
Para demostrar por qué el diseño del modelo importa, entrenamos una regresión logística **solo con los datos demográficos y antecedentes**, sin los ítems A1–A10. Si el resultado cae mucho, confirmamos que el poder predictivo viene de los ítems del cuestionario y no de las demás variables.

In [ ]:
solo_demograficas = ["age_mons", "sex_m", "jaundice", "family_mem_with_asd"]
pipe_demo = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), ["age_mons"])],
                                                 remainder="passthrough")),
                      ("clf", LogisticRegression(max_iter=1000, random_state=SEMILLA))])
pipe_demo.fit(X_train[solo_demograficas], y_train)
auc_demo = roc_auc_score(y_test, pipe_demo.predict_proba(X_test[solo_demograficas])[:, 1])

print(f"ROC-AUC con solo datos demográficos: {auc_demo:.3f}")
print(f"ROC-AUC del mejor modelo (con ítems): {tabla_test.loc[nombre_mejor, 'ROC-AUC']:.3f}")

<a id="s10"></a>
## 10. Importancia de las variables

Usamos la **importancia por permutación**: se mezclan al azar los valores de una variable y se mide cuánto empeora el modelo. Mientras más empeora, más importante era esa variable.

In [ ]:
imp = permutation_importance(mejor_modelo, X_test, y_test, scoring="roc_auc",
                             n_repeats=20, random_state=SEMILLA)

tabla_imp = (pd.DataFrame({"variable": VARIABLES, "importancia": imp.importances_mean})
             .sort_values("importancia", ascending=False).reset_index(drop=True))
display(tabla_imp.round(4))

plt.figure(figsize=(8, 5))
sns.barplot(data=tabla_imp, x="importancia", y="variable", color="#4C9BE8")
plt.title(f"Importancia por permutación – {nombre_mejor}")
plt.xlabel("Caída del ROC-AUC al mezclar la variable")
plt.show()

<a id="s11"></a>
## 11. Predicciones

### 11.1 Predicciones sobre el conjunto de prueba
Mostramos 15 niños del conjunto de prueba con su clase real, la predicha y la probabilidad estimada.

In [ ]:
prob_test = mejor_modelo.predict_proba(X_test)[:, 1]
tabla_pred = X_test.copy()
tabla_pred["real"] = y_test.values
tabla_pred["predicho"] = pred_mejor
tabla_pred["prob_rasgos_%"] = (prob_test * 100).round(1)
tabla_pred["acierto"] = np.where(tabla_pred["real"] == tabla_pred["predicho"], "✅", "❌")

tabla_pred[["age_mons", "sex_m", "real", "predicho", "prob_rasgos_%", "acierto"]].head(15)

### 11.2 Predicción de un caso nuevo
Simulamos el cuestionario de un niño que no está en el conjunto de datos. Puedes cambiar los valores y volver a ejecutar la celda (1 = señal de alerta, 0 = respuesta típica).

In [ ]:
caso_nuevo = pd.DataFrame([{
    "a1": 1, "a2": 1, "a3": 0, "a4": 1, "a5": 0,
    "a6": 1, "a7": 0, "a8": 1, "a9": 1, "a10": 0,
    "age_mons": 24, "sex_m": 1, "jaundice": 0, "family_mem_with_asd": 0}])

prob_nuevo = mejor_modelo.predict_proba(caso_nuevo[VARIABLES])[0, 1]
print(f"Probabilidad estimada de rasgos TEA: {prob_nuevo:.1%}")
print("Resultado:", "Se sugiere una evaluación profesional" if prob_nuevo >= 0.5
      else "Probabilidad baja según el modelo (continuar el seguimiento habitual)")

<a id="s12"></a>
## 12. Explicación de las variables

| Variable | Tipo | Descripción | Rol en el modelo |
|---|---|---|---|
| `a1` … `a10` | Binaria (0/1) | Respuestas a los 10 ítems del cuestionario Q-CHAT-10. **1 = señal de alerta**, 0 = respuesta típica | Predictora |
| `age_mons` | Numérica | Edad del niño en meses | Predictora (estandarizada) |
| `sex_m` | Binaria (0/1) | Sexo del niño: 1 = niño, 0 = niña | Predictora |
| `jaundice` | Binaria (0/1) | Si el niño tuvo ictericia al nacer | Predictora |
| `family_mem_with_asd` | Binaria (0/1) | Si hay un familiar con diagnóstico de TEA | Predictora |
| `qchat_10_score` | Numérica (0–10) | Suma de los ítems A1–A10 | **Excluida** (fuga de información) |
| `ethnicity` | Categórica | Etnicidad declarada | **Excluida** (decisión ética) |
| `who_completed_the_test` | Categórica | Quién respondió el cuestionario | **Excluida** (sin valor predictivo defendible) |
| `case_no` | Identificador | Número de caso | **Excluida** (identificador) |
| `asd` | Binaria (0/1) | Clase final: 1 = presenta rasgos TEA, 0 = no | **Variable objetivo** |

<a id="s13"></a>
## 13. Simulador interactivo en HTML

Para que el modelo pueda **usarse sin Python**, exportamos los parámetros de la regresión logística (intercepto, coeficientes y el escalado de la edad) y los incrustamos en una página HTML con JavaScript. Al mover los controles, el navegador calcula la probabilidad con la misma fórmula del modelo:

$$ p = \frac{1}{1 + e^{-(b_0 + \sum_i w_i x_i)}} $$

La celda siguiente:
1. Extrae los parámetros del modelo entrenado.
2. Verifica que el cálculo manual coincide con `predict_proba` de scikit-learn.
3. Genera el archivo `simulador_asd.html` y lo muestra dentro del cuaderno.

In [ ]:
# ===== 1) Extraemos los parámetros de la regresión logística =====
modelo_logit = pipelines["Regresión logística"]
prep = modelo_logit.named_steps["prep"]
clf = modelo_logit.named_steps["clf"]

# Nombres de columnas en el orden que ve el modelo (quitamos prefijos "num__" y "remainder__")
nombres = [n.split("__")[1] for n in prep.get_feature_names_out()]
escalador = prep.named_transformers_["num"]

parametros = {
    "intercepto": float(clf.intercept_[0]),
    "nombres": nombres,
    "coef": [float(c) for c in clf.coef_[0]],
    "media": {"age_mons": float(escalador.mean_[0])},
    "escala": {"age_mons": float(escalador.scale_[0])},
}

# ===== 2) Verificación: cálculo manual vs scikit-learn =====
def probabilidad_manual(fila, p):
    z = p["intercepto"]
    for nombre, w in zip(p["nombres"], p["coef"]):
        x = float(fila[nombre])
        if nombre in p["media"]:
            x = (x - p["media"][nombre]) / p["escala"][nombre]
        z += w * x
    return 1 / (1 + np.exp(-z))

p_manual = probabilidad_manual(caso_nuevo.iloc[0], parametros)
p_sklearn = modelo_logit.predict_proba(caso_nuevo[VARIABLES])[0, 1]
print(f"Probabilidad manual : {p_manual:.6f}")
print(f"Probabilidad sklearn: {p_sklearn:.6f}")
assert abs(p_manual - p_sklearn) < 1e-6, "Los cálculos no coinciden"
print("✅ El simulador reproduce exactamente al modelo")

In [ ]:
# ===== 3) Construimos la página HTML del simulador =====
edad_min, edad_max = int(df["age_mons"].min()), int(df["age_mons"].max())

# Controles para los 10 ítems (0 = típico, 1 = señal de alerta)
controles_items = "".join(
    f'<label>Ítem A{i}<select id="a{i}"><option value="0">0 – Respuesta típica</option>'
    f'<option value="1">1 – Señal de alerta</option></select></label>'
    for i in range(1, 11))

PLANTILLA = """
<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Simulador de tamizaje</title>
<style>
  body{font-family:Arial,Helvetica,sans-serif;margin:0;padding:16px;background:#f5f7fb;color:#222}
  h2{margin:0 0 4px} p.sub{margin:0 0 14px;color:#555;font-size:14px}
  .grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px}
  label{display:flex;flex-direction:column;font-size:13px;gap:4px;background:#fff;padding:8px;border-radius:8px;border:1px solid #dde3ee}
  select,input[type=range]{width:100%}
  .resultado{margin-top:16px;background:#fff;border-radius:10px;padding:14px;border:1px solid #dde3ee}
  .barra{height:22px;background:#e6ebf3;border-radius:11px;overflow:hidden}
  #relleno{height:100%;width:0;background:#4C9BE8;transition:width .3s,background .3s}
  #porcentaje{font-size:28px;font-weight:bold;margin:6px 0}
  .aviso{margin-top:10px;font-size:12px;color:#8a4b00;background:#fff4e0;padding:8px;border-radius:8px}
</style></head>
<body>
<h2>🧠 Simulador de tamizaje de rasgos TEA</h2>
<p class="sub">Ajusta los datos del niño. El cálculo se hace en tu navegador con los parámetros del modelo de regresión logística.</p>
<div class="grid">
  __CONTROLES_ITEMS__
  <label>Edad: <b id="edadTxt"></b> meses
    <input type="range" id="age_mons" min="__EDAD_MIN__" max="__EDAD_MAX__" value="24"></label>
  <label>Sexo<select id="sex_m"><option value="1">Niño</option><option value="0">Niña</option></select></label>
  <label>Ictericia al nacer<select id="jaundice"><option value="0">No</option><option value="1">Sí</option></select></label>
  <label>Familiar con TEA<select id="family_mem_with_asd"><option value="0">No</option><option value="1">Sí</option></select></label>
</div>
<div class="resultado">
  <div>Probabilidad estimada de rasgos TEA</div>
  <div id="porcentaje">–</div>
  <div class="barra"><div id="relleno"></div></div>
  <div id="mensaje" style="margin-top:8px;font-size:14px"></div>
</div>
<div class="aviso">⚠️ Herramienta educativa de apoyo. No es un diagnóstico: cualquier resultado debe ser valorado por profesionales de la salud.</div>
<script>
const M = __MODELO__;
function calcular(){
  let z = M.intercepto;
  M.nombres.forEach(function(n, i){
    let x = parseFloat(document.getElementById(n).value);
    if (M.media[n] !== undefined){ x = (x - M.media[n]) / M.escala[n]; }
    z += M.coef[i] * x;
  });
  const p = 1 / (1 + Math.exp(-z));
  const pct = (p * 100).toFixed(1);
  document.getElementById("edadTxt").textContent = document.getElementById("age_mons").value;
  document.getElementById("porcentaje").textContent = pct + " %";
  const barra = document.getElementById("relleno");
  barra.style.width = pct + "%";
  barra.style.background = p >= 0.5 ? "#F28C6B" : "#4C9BE8";
  document.getElementById("mensaje").textContent = p >= 0.5
    ? "Se sugiere una evaluación profesional."
    : "Probabilidad baja según el modelo; continuar el seguimiento habitual.";
}
document.querySelectorAll("select, input").forEach(function(el){
  el.addEventListener("input", calcular);
});
calcular();
</script>
</body></html>
"""

documento_html = (PLANTILLA
                  .replace("__CONTROLES_ITEMS__", controles_items)
                  .replace("__EDAD_MIN__", str(edad_min))
                  .replace("__EDAD_MAX__", str(edad_max))
                  .replace("__MODELO__", json.dumps(parametros)))

# Guardamos el archivo para publicarlo en el repositorio o en Netlify
with open("simulador_asd.html", "w", encoding="utf-8") as archivo:
    archivo.write(documento_html)
print("Archivo generado: simulador_asd.html ✅")

# Lo mostramos dentro del cuaderno usando un iframe
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.escape(documento_html, quote=True)}" '
             f'width="100%" height="760" style="border:1px solid #ccc;border-radius:8px"></iframe>'))

**Cómo usar el simulador fuera del cuaderno:** abre `simulador_asd.html` en cualquier navegador, o súbelo a Netlify o GitHub Pages para incluirlo como parte de tu aplicación web. Si lo integras en una app de Streamlit, puedes cargarlo con `streamlit.components.v1.html`.

In [ ]:
# Guardamos el mejor modelo entrenado para reutilizarlo en la aplicación web (Streamlit)
joblib.dump({"modelo": mejor_modelo, "variables": VARIABLES, "nombre": nombre_mejor},
            "modelo_tamizaje.joblib")
print("Modelo guardado en: modelo_tamizaje.joblib ✅")

# Cerramos la conexión a la base de datos
conexion.close()

<a id="s14"></a>
## 14. Conclusiones, limitaciones y próximos pasos

### 14.1 Resumen automático de resultados
La siguiente celda redacta el resumen con los números reales de tu ejecución.

In [ ]:
m = tabla_test.loc[nombre_mejor]
print(f"Fuente de datos usada: {FUENTE}")
print(f"Total de registros: {len(df)} | Clase 1 (con rasgos): {df['asd'].mean():.1%}")
print(f"Mejor modelo: {nombre_mejor}")
print(f"  Accuracy {m['Accuracy']:.3f} | Precisión {m['Precisión']:.3f} | "
      f"Recall {m['Recall']:.3f} | F1 {m['F1']:.3f} | ROC-AUC {m['ROC-AUC']:.3f}")
print(f"Variable más importante: {tabla_imp.loc[0, 'variable']}")
print(f"ROC-AUC solo con datos demográficos: {auc_demo:.3f} (referencia de la contribución de los ítems)")

### 14.2 Conclusiones
1. Se construyó un flujo completo de Machine Learning: **ELT → Data Profiling → EDA → modelado → evaluación → simulador**, reproducible de principio a fin.
2. Se compararon tres modelos con validación cruzada y se seleccionó el mejor con métricas apropiadas para un contexto de salud, dando prioridad al *recall*.
3. Los ítems del cuestionario concentran el poder predictivo, mientras que las variables demográficas aportan muy poco (ver el experimento de la sección 9.2).
4. El **simulador en HTML** permite que una persona sin conocimientos de programación explore el modelo, lo que acerca el análisis de datos a un uso real.
5. Todo el proceso es transparente: cada decisión (variables excluidas, métricas, modelo elegido) está justificada y puede sustentarse.

### 14.3 Limitaciones
- **Fuga de información en la etiqueta:** en el dataset original, la clase se deriva de la puntuación del cuestionario. Por eso los modelos alcanzan un rendimiento muy alto: aprenden a reproducir esa regla y no a "diagnosticar". Esto debe declararse en el documento técnico.
- **Muestra limitada y no representativa:** los datos provienen de un conjunto específico de tamizajes y pueden no representar a otras poblaciones, incluida la dominicana.
- **No es un diagnóstico:** la clase indica rasgos en un tamizaje, no una evaluación clínica.
- Si se ejecutó con datos **sintéticos**, los resultados son solo demostrativos.

### 14.4 Próximos pasos
- Validar el modelo con datos locales y con apoyo de profesionales de salud y educación.
- Ampliar el proyecto a otras condiciones del neurodesarrollo (por ejemplo, TDAH o dificultades del aprendizaje).
- Publicar la aplicación web en Streamlit o Netlify e integrar el simulador.
- Incorporar técnicas de explicabilidad más completas (por ejemplo, SHAP) y análisis de equidad por grupos.

<a id="s15"></a>
## 15. Referencias

> Verifica y completa las referencias (año, URL, fecha de consulta) antes de entregarlas en formato APA.

- Thabtah, F. (2018). *Autism screening data for toddlers* [Conjunto de datos]. Kaggle.
- Allison, C., Auyeung, B., & Baron-Cohen, S. (2012). Toward brief "red flags" for autism screening: The Short Autism Spectrum Quotient and the Short Quantitative Checklist for Autism in toddlers. *Journal of the American Academy of Child & Adolescent Psychiatry, 51*(2), 202–212.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Brugman, S. (2019). *pandas-profiling / ydata-profiling: Automated exploratory data analysis* [Software].